# Analisis Temporal de Precios - TFG

Este notebook usa los datos procesados para analizar patrones de dynamic pricing en portatiles de Amazon, PcComponentes y El Corte Inglés.

Bloques principales:
1. Evolucion diaria de precios por plataforma.
2. Ranking de productos con mayor variacion de precio.
3. Resumen ejecutivo para memoria del TFG.

In [9]:
from pathlib import Path
import pandas as pd
import plotly.express as px

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"
candidatos = sorted(PROCESSED_DIR.glob("precios_portatiles_procesado_*.csv"))
if not candidatos:
    raise FileNotFoundError("No hay CSV procesados en data/processed")

latest_csv = candidatos[-1]
print(f"CSV procesado cargado: {latest_csv}")

CSV procesado cargado: /Users/david/Documents/scraper/data/processed/precios_portatiles_procesado_20260420_2351.csv


In [10]:
import pandas as pd
from pathlib import Path

# 1. Buscamos el último archivo procesado que creaste con el ETL
processed_files = sorted(Path("../data/processed").glob("precios_portatiles_procesado_*.csv"))
latest_file = processed_files[-1]
df = pd.read_csv(latest_file)

# 2. Creamos la columna 'fecha_dia' que faltaba y daba error
df['fecha_extraccion'] = pd.to_datetime(df['fecha_extraccion'])
df['fecha_dia'] = df['fecha_extraccion'].dt.date 

# 3. FILTRO DE CALIDAD: Aquí es donde "quitamos" los monitores e iPads de la gráfica
# Pero solo para el análisis, en tu CSV siguen estando guardados.
palabras_ruido = ['monitor', 'tablet', 'impresora', 'all in one', 'ipad', 'tab', 'smart monitor']
df = df[~df['nombre'].str.lower().str.contains('|'.join(palabras_ruido), na=False)]

print(f"✅ ¡Listo! Analizando el archivo: {latest_file.name}")
print(f"Plataformas en la gráfica: {df['plataforma'].unique()}")
display(df.head(5))

✅ ¡Listo! Analizando el archivo: precios_portatiles_procesado_20260420_2351.csv
Plataformas en la gráfica: <StringArray>
['Amazon', 'ElCorteIngles', 'PcComponentes']
Length: 3, dtype: str


,nombre,precio_actual,precio_original,descuento,valoracion,plataforma,fecha,precio_actual_num,precio_original_num,descuento_pct,fecha_extraccion,anio,mes,dia,fecha_dia
0,Ordenador portátil 14 pulgadas Dual-Core hasta...,"205,.99",NaN,NaN,"4,1 de 5 estrellas",Amazon,2026-04-20 23:20,205.99,NaN,NaN,2026-04-20 23:20:00,2026,4,20,2026-04-20
1,2026 Ordenador portátil 15.6 Pulgadas pc portá...,"209,.99",NaN,NaN,"5,0 de 5 estrellas",Amazon,2026-04-20 23:20,209.99,NaN,NaN,2026-04-20 23:20:00,2026,4,20,2026-04-20
2,PINSTONE Ordenador Portátil de 14 Pulgadas Cel...,"214,.00",NaN,NaN,"4,8 de 5 estrellas",Amazon,2026-04-20 23:20,214.00,NaN,NaN,2026-04-20 23:20:00,2026,4,20,2026-04-20
3,"BMAX Ordenador Portátil 15.6"" FHD Celeron N95 ...","256,.49","289,99 €",NaN,"4,6 de 5 estrellas",Amazon,2026-04-20 23:20,256.49,289.99,11.55,2026-04-20 23:20:00,2026,4,20,2026-04-20
4,ASUS Chromebook CX1505CTA - Ordenador Portátil...,"259,.00",NaN,NaN,"4,2 de 5 estrellas",Amazon,2026-04-20 23:20,259.00,NaN,NaN,2026-04-20 23:20:00,2026,4,20,2026-04-20


In [11]:
evolucion = (
    df.groupby(["fecha_dia", "plataforma"], as_index=False)
      .agg(
          precio_medio=("precio_actual_num", "mean"),
          precio_mediano=("precio_actual_num", "median"),
          n_productos=("nombre", "count")
      )
)

display(evolucion.sort_values(["fecha_dia", "plataforma"]))

fig = px.line(
    evolucion,
    x="fecha_dia",
    y="precio_medio",
    color="plataforma",
    markers=True,
    title="Evolución diaria del precio medio por plataforma (Amazon, PcComponentes, El Corte Inglés)"
)
fig.update_layout(xaxis_title="Fecha", yaxis_title="Precio medio (EUR)")
fig.show()

,fecha_dia,plataforma,precio_medio,precio_mediano,n_productos
0,2026-04-20,Amazon,398.2252,368.67,25
1,2026-04-20,ElCorteIngles,965.0000,949.00,5
2,2026-04-20,PcComponentes,916.1584,899.00,25


In [12]:
# Firma de producto para agrupar el mismo item a lo largo del tiempo
df["producto_id"] = (
    df["nombre"].astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str[:90]
)

variacion = (
    df.groupby(["plataforma", "producto_id"], as_index=False)
      .agg(
          observaciones=("precio_actual_num", "count"),
          precio_min=("precio_actual_num", "min"),
          precio_max=("precio_actual_num", "max")
      )
)

variacion["variacion_abs"] = (variacion["precio_max"] - variacion["precio_min"]).round(2)
variacion["variacion_pct"] = ((variacion["variacion_abs"] / variacion["precio_min"]) * 100).round(2)

ranking = variacion.sort_values("variacion_abs", ascending=False).head(20)
display(ranking)

fig_rank = px.bar(
    ranking.head(10),
    x="variacion_abs",
    y="producto_id",
    color="plataforma",
    orientation="h",
    title="Top 10 productos con mayor variación absoluta (Amazon, PcComponentes, El Corte Inglés)"
)
fig_rank.update_layout(xaxis_title="Variacion absoluta (EUR)", yaxis_title="Producto (id abreviado)")
fig_rank.show()

,plataforma,producto_id,observaciones,precio_min,precio_max,variacion_abs,variacion_pct
0,Amazon,"15.6 pulgadas ordenador portatil, procesador n...",1,289.99,289.99,0.0,0.0
41,PcComponentes,portátil alurin flex advance amd ryzen 7-5825u...,1,509.99,509.99,0.0,0.0
30,PcComponentes,acer aspire go 15 ag15-42p-r26t 15.6″ amd ryze...,1,449.00,449.00,0.0,0.0
31,PcComponentes,apple macbook air apple m4 10 núcleos/16 gb/25...,1,899.00,899.00,0.0,0.0
32,PcComponentes,apple macbook air apple m4 10 núcleos/16 gb/51...,1,1219.00,1219.00,0.0,0.0
33,PcComponentes,asus tuf gaming fx608jmr-rv003 intel core i7-1...,1,1299.00,1299.00,0.0,0.0
34,PcComponentes,asus vivobook go 15 e1504fa-bq2447 amd ryzen 5...,1,444.99,444.99,0.0,0.0
35,PcComponentes,hp omen 16-ap0007ns amd ryzen ai 7 350 32gb 1t...,1,1429.00,1429.00,0.0,0.0
36,PcComponentes,msi cyborg 15 b2rwfkg-201xes intel core 7 240h...,1,1249.00,1249.00,0.0,0.0
37,PcComponentes,"pccom revolt 5060 intel core i7-14650hx 16"" /1...",1,1299.99,1299.99,0.0,0.0


In [13]:
resumen = (
    df.groupby("plataforma", as_index=False)
      .agg(
          productos=("nombre", "count"),
          precio_medio=("precio_actual_num", "mean"),
          precio_mediano=("precio_actual_num", "median"),
          precio_min=("precio_actual_num", "min"),
          precio_max=("precio_actual_num", "max"),
          descuento_medio_pct=("descuento_pct", "mean")
      )
)

resumen = resumen.round(2).sort_values("precio_medio")
display(resumen)

print("Resumen ejecutivo automatico (Amazon, PcComponentes, El Corte Inglés):")
print(f"- Muestra total analizada: {len(df)} observaciones")
for _, row in resumen.iterrows():
    print(
        f"- {row['plataforma']}: n={int(row['productos'])}, precio medio={row['precio_medio']} EUR, "
        f"rango=[{row['precio_min']}, {row['precio_max']}] EUR"
    )

if (variacion['variacion_abs'] > 0).any():
    top = variacion.sort_values('variacion_abs', ascending=False).iloc[0]
    print(
        f"- Mayor variacion detectada: {top['variacion_abs']} EUR en {top['plataforma']} (producto_id abreviado)."
    )
else:
    print("- Aun no hay variacion temporal significativa: se necesita acumular mas dias de extraccion.")

,plataforma,productos,precio_medio,precio_mediano,precio_min,precio_max,descuento_medio_pct
0,Amazon,25,398.23,368.67,205.99,719.0,13.67
2,PcComponentes,25,916.16,899.00,269.99,1629.0,19.00
1,ElCorteIngles,5,965.00,949.00,779.00,1199.0,12.77


Resumen ejecutivo automatico (Amazon, PcComponentes, El Corte Inglés):
- Muestra total analizada: 55 observaciones
- Amazon: n=25, precio medio=398.23 EUR, rango=[205.99, 719.0] EUR
- PcComponentes: n=25, precio medio=916.16 EUR, rango=[269.99, 1629.0] EUR
- ElCorteIngles: n=5, precio medio=965.0 EUR, rango=[779.0, 1199.0] EUR
- Aun no hay variacion temporal significativa: se necesita acumular mas dias de extraccion.
